[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 Easy: Kaiming Initialization

*Core Ops & Layers*
Implement **Kaiming (He) initialization**, the standard for ReLU networks.

$$W \sim \mathcal{N}\!\left(0,\ \sigma^2\right), \qquad
\sigma = \sqrt{\frac{2}{\text{fan\_in}}}$$

### Signature
```python
def kaiming_init(key, weight):
    ...  # -> a new array shaped like `weight`
```

### Rules
- Draw from a **normal** distribution with mean 0
- `std = sqrt(2 / fan_in)`
- `fan_in` is the input dimension, which is `weight.shape[0]` in both cases:
  a 2-D Flax kernel is `(in_features, out_features)`, and a 1-D array has only
  one axis
- Do not use `nnx.initializers` or `jax.nn.initializers`

### Where the 2 comes from
For a linear layer $y = Wx$ with independent zero-mean weights,
$\mathrm{Var}(y) = \text{fan\_in} \cdot \mathrm{Var}(W) \cdot \mathrm{Var}(x)$.
To keep the variance from shrinking or exploding layer over layer you want that
factor to be 1, giving Xavier's $\mathrm{Var}(W) = 1/\text{fan\_in}$.

But ReLU zeroes half its inputs, which halves the output variance. Kaiming
compensates by doubling: $\mathrm{Var}(W) = 2/\text{fan\_in}$. That single
factor of 2 is the entire difference from Xavier — and it is what made it
possible to train very deep ReLU networks without careful layer-wise
pre-training.

Use Kaiming with ReLU/GELU/SiLU, Xavier with tanh/sigmoid.

### ⚠️ Two things JAX forces here
1. **A key argument.** PyTorch calls `weight.normal_(0, std)`, drawing from a
   hidden global RNG. JAX has no global RNG, so randomness is explicit — the
   key is the first argument, by convention.
2. **A return value.** `normal_` fills in place; JAX arrays are immutable, so
   this returns a new array instead.

`weight` is therefore only used for its **shape and dtype**. Passing the array
rather than the shape keeps the call site looking like the PyTorch original.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def kaiming_init(key, weight):
    """Kaiming-normal values shaped like `weight`.

    Args:
        key:    a jax.random key
        weight: array whose shape (and dtype) the result should match

    Returns:
        A NEW array shaped like `weight`, drawn from N(0, sqrt(2/fan_in)).
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

w = jnp.zeros((256, 64))
out = kaiming_init(jax.random.key(0), w)

print("shape:", out.shape)
print("mean :", float(out.mean()), "(~0)")
print("std  :", float(out.std()), "vs target", float(jnp.sqrt(2 / 256)))

# Variance is preserved through a ReLU stack — that is the whole point.
x = jax.random.normal(jax.random.key(1), (1000, 256))
for layer in range(5):
    w = kaiming_init(jax.random.key(layer + 2), jnp.zeros((x.shape[-1], 256)))
    x = jax.nn.relu(x @ w)
    print(f"  after layer {layer}: std {float(x.std()):.3f}")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("weight_init")

# hint("weight_init")      # stuck? nudge without the answer
# solution("weight_init")  # spoiler: the reference implementation
# status()                 # your dashboard across all problems